# Sesión 5 — SQL de negocio y análisis operativo

**Mensaje central:** Silver nos da datos confiables; SQL nos permite convertirlos en evidencia, insights y decisiones.

En esta sesión trabajaremos sobre las tablas Silver creadas en la Sesión 4. El objetivo no es crear Gold todavía, sino dejar consultas candidatas y entender bien granularidad, joins, agregaciones y riesgo de doble conteo.

## 0. Contexto técnico esperado

Catálogo: `workspace`

Tablas principales:

- `workspace.lumi_silver.orders_clean`
- `workspace.lumi_silver.order_items_clean`
- `workspace.lumi_silver.payments_clean`
- `workspace.lumi_silver.reviews_clean`
- `workspace.lumi_silver.products_clean`
- `workspace.lumi_silver.sellers_clean`
- `workspace.bagazo_silver.operacion_ingenios_clean`
- `workspace.control.quality_summary_sesion_04`

In [0]:
%sql
USE CATALOG workspace;
SHOW SCHEMAS;

## 1. Salud de Silver

Abrimos la clase con gobierno de datos. Antes de analizar, revisamos qué tan confiables son las tablas que vamos a consultar.

In [0]:
%sql
SELECT *
FROM workspace.control.quality_summary_sesion_04
ORDER BY dataset, tabla;

In [0]:
%sql
SELECT
  estado_calidad,
  COUNT(*) AS tablas
FROM workspace.control.quality_summary_sesion_04
GROUP BY estado_calidad
ORDER BY tablas DESC;

In [0]:
%sql
SELECT
  dataset, tabla, filas, columnas, duplicados_clave, reglas_fallidas, estado_calidad, observaciones
FROM workspace.control.quality_summary_sesion_04
WHERE estado_calidad <> 'OK'
ORDER BY dataset, tabla;

### Reflexión guiada

`reviews_clean` quedó en estado **REVISAR** por duplicados de clave. Esto no bloquea el análisis, pero sí cambia la forma correcta de hacer joins. Cuando crucemos reviews con pedidos o ítems, primero agregaremos reviews por `order_id`.

## 2. Granularidad: la regla que evita métricas infladas

- `orders_clean`: una fila por pedido.
- `payments_clean`: consolidada por `order_id`.
- `order_items_clean`: una fila por ítem de pedido.
- `reviews_clean`: puede tener más de una fila por pedido; requiere agregación previa.
- `operacion_ingenios_clean`: una fila por fecha e ingenio.

**Regla de oro:** si dos tablas tienen granularidad diferente, agrega primero y une después.

# Bloque A — Lumi Commerce Lakehouse

## 3. Ventas por mes

Usamos `order_items_clean` porque la venta está a nivel de ítem. Unimos con `orders_clean` para filtrar pedidos entregados y agrupar por mes.

**TODO pedagógico 1:** cambia temporalmente el filtro `order_status = 'delivered'` por otro estado y compara el resultado.

In [0]:
%sql
WITH ventas_item AS (
  SELECT
    o.year_month,
    oi.order_id,
    oi.order_item_id,
    oi.total_item_value
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  WHERE o.order_status = 'delivered'
)
SELECT
  year_month,
  COUNT(DISTINCT order_id) AS pedidos_entregados,
  COUNT(*) AS items_vendidos,
  ROUND(SUM(total_item_value), 2) AS venta_total_items,
  ROUND(AVG(total_item_value), 2) AS valor_promedio_item
FROM ventas_item
GROUP BY year_month
ORDER BY year_month;

In [0]:
%sql
WITH ventas_item AS (
  SELECT
    o.year_month,
    oi.order_id,
    oi.order_item_id,
    oi.total_item_value
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  WHERE o.order_status = 'invoiced'
)
SELECT
  year_month,
  COUNT(DISTINCT order_id) AS pedidos_entregados,
  COUNT(*) AS items_vendidos,
  ROUND(SUM(total_item_value), 2) AS venta_total_items,
  ROUND(AVG(total_item_value), 2) AS valor_promedio_item
FROM ventas_item
GROUP BY year_month
ORDER BY year_month;

## 4. Ticket promedio por pedido

Aquí usamos `payments_clean`, que ya quedó consolidada por pedido en Silver. Por eso podemos calcular ticket promedio sin unir contra ítems.

**TODO pedagógico 2:** identifica por qué usamos `COUNT(DISTINCT o.order_id)` y no simplemente `COUNT(*)` en análisis con joins.

### Respuesta TODO Pedagógico 2

**¿Por qué `COUNT(DISTINCT o.order_id)` en lugar de `COUNT(*)`?**

Cuando hacemos un **JOIN**, el resultado puede tener más filas que la tabla original. Si el pedido 100 tiene 2 registros de pago, el JOIN genera 2 filas:

- `COUNT(*)` = 2 ❌ (cuenta el pedido 100 dos veces)
- `COUNT(DISTINCT order_id)` = 1 ✅ (cuenta solo pedidos únicos)

**Regla:**
- `COUNT(*)` → número de **filas** después del JOIN
- `COUNT(DISTINCT id)` → número de **entidades únicas** (pedidos, clientes)

Usamos `COUNT(DISTINCT o.order_id)` para contar **pedidos únicos**, no filas. Aunque `payments_clean` está consolidada, el DISTINCT protege contra duplicados no detectados.

In [0]:
%sql
SELECT
  o.year_month,
  COUNT(DISTINCT o.order_id) AS pedidos_entregados,
  ROUND(SUM(COALESCE(p.total_payment_value, 0)), 2) AS valor_pagado_total,
  ROUND(AVG(COALESCE(p.total_payment_value, 0)), 2) AS ticket_promedio_pedido
FROM workspace.lumi_silver.orders_clean o
LEFT JOIN workspace.lumi_silver.payments_clean p
  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY o.year_month
ORDER BY o.year_month;

## 5. Pedidos por estado

In [0]:
%sql
SELECT
  order_status,
  COUNT(*) AS pedidos,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM workspace.lumi_silver.orders_clean
GROUP BY order_status
ORDER BY pedidos DESC;

## 6. Ventas por categoría

Cuidamos la granularidad: ventas por categoría se calcula desde ítems, no desde pagos.

In [0]:
%sql
WITH ventas_categoria AS (
  SELECT
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria,
    oi.order_id,
    oi.order_item_id,
    oi.total_item_value
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  WHERE o.order_status = 'delivered'
)
SELECT
  categoria,
  COUNT(DISTINCT order_id) AS pedidos,
  COUNT(*) AS items,
  ROUND(SUM(total_item_value), 2) AS venta_total,
  ROUND(AVG(total_item_value), 2) AS valor_promedio_item
FROM ventas_categoria
GROUP BY categoria
ORDER BY venta_total DESC
LIMIT 20;

## 7. Métodos de pago

In [0]:
%sql
SELECT
  main_payment_type,
  COUNT(*) AS pedidos,
  ROUND(SUM(total_payment_value), 2) AS valor_total_pagado,
  ROUND(AVG(total_payment_value), 2) AS ticket_promedio
FROM workspace.lumi_silver.payments_clean
GROUP BY main_payment_type
ORDER BY valor_total_pagado DESC;

## 8. Reviews por categoría

Antes de cruzar reviews con categorías, agregamos reviews por pedido.

**TODO pedagógico 3:** en la CTE `reviews_by_order`, identifica la métrica que representa satisfacción promedio por pedido.

### Respuesta TODO Pedagógico 3

**Métrica que representa satisfacción promedio por pedido:**

`review_promedio_pedido` = `ROUND(AVG(review_score), 2)`



In [0]:
%sql
WITH reviews_by_order AS (
  SELECT
    order_id,
    ROUND(AVG(review_score), 2) AS review_promedio_pedido,
    MAX(CASE WHEN is_low_review THEN 1 ELSE 0 END) AS tiene_review_bajo
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
category_orders AS (
  SELECT DISTINCT
    oi.order_id,
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria
  FROM workspace.lumi_silver.order_items_clean oi
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
)
SELECT
  co.categoria,
  COUNT(DISTINCT co.order_id) AS pedidos_con_categoria,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio,
  SUM(CASE WHEN r.tiene_review_bajo = 1 THEN 1 ELSE 0 END) AS pedidos_con_review_bajo
FROM category_orders co
LEFT JOIN reviews_by_order r
  ON co.order_id = r.order_id
GROUP BY co.categoria
HAVING COUNT(DISTINCT co.order_id) >= 50
ORDER BY review_promedio ASC, pedidos_con_categoria DESC
LIMIT 20;

## 9. Entregas tardías

In [0]:
%sql
SELECT
  year_month,
  COUNT(*) AS pedidos_entregados,
  SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS pedidos_tarde,
  ROUND(100.0 * SUM(CASE WHEN is_late THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_entrega_tarde,
  ROUND(AVG(delay_days), 2) AS demora_promedio_dias
FROM workspace.lumi_silver.orders_clean
WHERE order_status = 'delivered'
GROUP BY year_month
ORDER BY year_month;

## 10. Relación entre demora y review

In [0]:
%sql
WITH reviews_by_order AS (
  SELECT
    order_id,
    ROUND(AVG(review_score), 2) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
)
SELECT
  CASE
    WHEN o.delay_days <= 0 THEN 'A tiempo o antes'
    WHEN o.delay_days BETWEEN 1 AND 3 THEN '1 a 3 días tarde'
    WHEN o.delay_days BETWEEN 4 AND 7 THEN '4 a 7 días tarde'
    ELSE 'Más de 7 días tarde'
  END AS tramo_demora,
  COUNT(DISTINCT o.order_id) AS pedidos,
  ROUND(AVG(o.delay_days), 2) AS demora_promedio,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio
FROM workspace.lumi_silver.orders_clean o
LEFT JOIN reviews_by_order r
  ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
GROUP BY tramo_demora
ORDER BY demora_promedio;

## 11. Ranking de vendedores

In [0]:
%sql
SELECT
  s.seller_id,
  s.seller_state,
  COUNT(DISTINCT oi.order_id) AS pedidos,
  COUNT(*) AS items,
  ROUND(SUM(oi.total_item_value), 2) AS venta_total,
  ROUND(AVG(oi.total_item_value), 2) AS valor_promedio_item
FROM workspace.lumi_silver.order_items_clean oi
INNER JOIN workspace.lumi_silver.orders_clean o
  ON oi.order_id = o.order_id
LEFT JOIN workspace.lumi_silver.sellers_clean s
  ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_id, s.seller_state
ORDER BY venta_total DESC
LIMIT 20;

## 12. Categorías con alta venta y baja satisfacción

In [0]:
%sql
WITH ventas_categoria AS (
  SELECT
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria,
    COUNT(DISTINCT oi.order_id) AS pedidos,
    ROUND(SUM(oi.total_item_value), 2) AS venta_total
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  WHERE o.order_status = 'delivered'
  GROUP BY COALESCE(p.product_category_clean, 'sin_categoria')
),
reviews_by_order AS (
  SELECT order_id, AVG(review_score) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
category_orders AS (
  SELECT DISTINCT
    oi.order_id,
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria
  FROM workspace.lumi_silver.order_items_clean oi
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
),
reviews_categoria AS (
  SELECT
    co.categoria,
    ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio
  FROM category_orders co
  LEFT JOIN reviews_by_order r
    ON co.order_id = r.order_id
  GROUP BY co.categoria
)
SELECT
  v.categoria,
  v.pedidos,
  v.venta_total,
  r.review_promedio,
  CASE
    WHEN v.venta_total >= 100000 AND r.review_promedio < 4.0 THEN 'Alta prioridad'
    WHEN v.venta_total >= 50000 AND r.review_promedio < 4.0 THEN 'Revisar'
    ELSE 'Monitorear'
  END AS prioridad_analitica
FROM ventas_categoria v
LEFT JOIN reviews_categoria r
  ON v.categoria = r.categoria
WHERE v.pedidos >= 100
ORDER BY prioridad_analitica, v.venta_total DESC, r.review_promedio ASC
LIMIT 25;

# Bloque B — Lluvia, caña y bagazo

## 13. Lectura operativa del caso Bagazo

La tabla quedó en granularidad limpia: una fila por `fecha` e `ingenio`. Aquí los ceros operativos no se tratan automáticamente como errores: se interpretan con comentarios y banderas.

In [0]:
%sql
SELECT *
FROM workspace.bagazo_silver.operacion_ingenios_clean
LIMIT 20;

## 14. Lluvia y bagazo por mes

In [0]:
%sql
SELECT
  year_month,
  ROUND(AVG(lluvia_mm), 2) AS lluvia_promedio_mm,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton,
  ROUND(AVG(cana_molida_ton), 2) AS cana_promedio_ton,
  COUNT(*) AS registros
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY year_month
ORDER BY year_month;

## 15. Caña y bagazo por ingenio

In [0]:
%sql
SELECT
  ingenio,
  ROUND(SUM(cana_molida_ton), 2) AS cana_total_ton,
  ROUND(SUM(bagazo_entregado_ton), 2) AS bagazo_total_ton,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton,
  SUM(CASE WHEN riesgo_bajo_bagazo THEN 1 ELSE 0 END) AS dias_riesgo_bajo
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio
ORDER BY bagazo_total_ton DESC;

## 16. Días con lluvia alta y bajo bagazo

In [0]:
%sql
SELECT
  fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton,
  lluvia_alta, riesgo_bajo_bagazo, tiene_comentario_operativo, comentario
FROM workspace.bagazo_silver.operacion_ingenios_clean
WHERE lluvia_alta = true OR riesgo_bajo_bagazo = true
ORDER BY fecha, ingenio
LIMIT 80;

## 17. Promedio de bagazo en días secos vs lluviosos

In [0]:
%sql
SELECT
  ingenio,
  CASE WHEN lluvia_alta THEN 'Día con lluvia alta' ELSE 'Día sin lluvia alta' END AS tipo_dia,
  COUNT(*) AS dias,
  ROUND(AVG(lluvia_mm), 2) AS lluvia_promedio_mm,
  ROUND(AVG(cana_molida_ton), 2) AS cana_promedio_ton,
  ROUND(AVG(bagazo_entregado_ton), 2) AS bagazo_promedio_ton
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio, CASE WHEN lluvia_alta THEN 'Día con lluvia alta' ELSE 'Día sin lluvia alta' END
ORDER BY ingenio, tipo_dia;

## 18. Correlación simple por ingenio

In [0]:
%sql
SELECT
  ingenio,
  COUNT(*) AS observaciones,
  ROUND(CORR(lluvia_mm, bagazo_entregado_ton), 4) AS corr_lluvia_bagazo,
  ROUND(CORR(cana_molida_ton, bagazo_entregado_ton), 4) AS corr_cana_bagazo,
  ROUND(CORR(lluvia_mm, cana_molida_ton), 4) AS corr_lluvia_cana
FROM workspace.bagazo_silver.operacion_ingenios_clean
GROUP BY ingenio
ORDER BY ingenio;

## 19. Días críticos explicados por comentarios operativos

In [0]:
%sql
SELECT
  fecha, ingenio, lluvia_mm, cana_molida_ton, bagazo_entregado_ton,
  es_mantenimiento, es_falta_cana_por_lluvia, es_paro, es_sin_recepcion_bagazo,
  comentario
FROM workspace.bagazo_silver.operacion_ingenios_clean
WHERE riesgo_bajo_bagazo = true
  AND tiene_comentario_operativo = true
ORDER BY fecha, ingenio
LIMIT 50;

# Cierre — De SQL a insight ejecutivo

**TODO pedagógico 4:** escribe un insight ejecutivo usando este formato:

```text
Insight:
Evidencia:
Impacto operativo o de negocio:
Recomendación:
Consulta SQL usada:
```

### Respuesta TODO Pedagógico 4: Insight Ejecutivo

---

**Insight 1: Demora en entregas deteriora significativamente la satisfacción del cliente**

**Evidencia:**  
Los pedidos entregados a tiempo tienen un review promedio de 4.23, mientras que pedidos con más de 7 días de demora caen a 3.71 (-0.52 puntos, -12.3%). Los pedidos con 4-7 días tarde promedian 4.03.

**Impacto operativo o de negocio:**  
- 12.3% de pérdida en satisfacción por entregas muy tardías
- Reviews bajos (<4.0) aumentan churn y reducen conversión en marketplace
- Categorías de alto valor con entregas tardías pierden ventaja competitiva

**Recomendación:**  
1. Priorizar mejora logística en categorías de alta venta (health_beauty, watches_gifts)
2. Implementar alertas automáticas para pedidos en riesgo de superar 3 días de demora
3. Ofrecer compensación proactiva (descuento futuro) a pedidos con +5 días de retraso

**Consulta SQL usada:** Celda 29 - Relación entre demora y review

---

**Insight 2: Lluvia alta reduce producción de bagazo en 23% promedio**

**Evidencia:**  
En días sin lluvia alta, los ingenios promedian 455-565 toneladas de bagazo. En días con lluvia alta, el promedio cae a 327-451 toneladas (-23% a -28% según ingenio). Correlación lluvia-bagazo: -0.25 a -0.33.

**Impacto operativo o de negocio:**  
- Reducción de 100-150 toneladas diarias de bagazo disponible para energía
- Necesidad de combustible alternativo (más caro) en períodos lluviosos
- Impacto en contratos de suministro energético con terceros

**Recomendación:**  
1. Crear buffer de inventario de bagazo para períodos lluviosos previsibles
2. Negociar contratos flexibles de combustible alternativo con proveedores
3. Integrar pronósticos meteorológicos en planificación semanal de producción
4. Excluir días de lluvia alta de modelos ML para forecasting de suministro

**Consulta SQL usada:** Celdas 38, 44, 46 - Lluvia y bagazo por mes, días secos vs lluviosos, correlaciones

# Retos finales

Los retos no bloquean el flujo principal. Úsalos para práctica individual o trabajo por grupos.

## Reto nivel 1
Identifica las 10 categorías con mayor review promedio o los 10 estados con más pedidos entregados.

## Reto nivel 2
Construye una CTE que combine ventas, demora y review promedio por categoría sin duplicar pagos ni pedidos.

## Reto consultor
Entrega 3 insights en formato ejecutivo, cada uno respaldado por una consulta SQL.

In [0]:
%sql
-- RETO NIVEL 1 - Parte 1: Top 10 categorías con mayor review promedio

WITH reviews_by_order AS (
  SELECT
    order_id,
    AVG(review_score) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
category_orders AS (
  SELECT DISTINCT
    oi.order_id,
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN workspace.lumi_silver.orders_clean o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  WHERE o.order_status = 'delivered'
)
SELECT
  co.categoria,
  COUNT(DISTINCT co.order_id) AS pedidos,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio,
  ROUND(MIN(r.review_promedio_pedido), 2) AS review_minimo,
  ROUND(MAX(r.review_promedio_pedido), 2) AS review_maximo
FROM category_orders co
LEFT JOIN reviews_by_order r
  ON co.order_id = r.order_id
GROUP BY co.categoria
HAVING COUNT(DISTINCT co.order_id) >= 50  -- Filtrar categorías con al menos 50 pedidos
ORDER BY review_promedio DESC, pedidos DESC
LIMIT 10;

In [0]:
%sql
-- RETO NIVEL 1 - Parte 2: Top 10 estados con más pedidos entregados

SELECT
  c.customer_state AS estado,
  COUNT(DISTINCT o.order_id) AS pedidos_entregados,
  COUNT(DISTINCT c.customer_id) AS clientes_unicos,
  ROUND(COUNT(DISTINCT o.order_id) * 100.0 / SUM(COUNT(DISTINCT o.order_id)) OVER(), 2) AS porcentaje_total_pedidos,
  ROUND(AVG(o.delay_days), 2) AS demora_promedio_dias
FROM workspace.lumi_silver.orders_clean o
INNER JOIN workspace.lumi_silver.customers_clean c
  ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY pedidos_entregados DESC
LIMIT 10;

In [0]:
%sql
-- RETO NIVEL 2: CTE que combine ventas, demora y review promedio por categoría
-- sin duplicar pagos ni pedidos

WITH reviews_by_order AS (
  -- Consolidar reviews por pedido (evita duplicados)
  SELECT
    order_id,
    AVG(review_score) AS review_promedio_pedido
  FROM workspace.lumi_silver.reviews_clean
  GROUP BY order_id
),
order_base AS (
  -- Base de pedidos entregados con información de demora
  SELECT
    o.order_id,
    o.delay_days,
    o.is_late
  FROM workspace.lumi_silver.orders_clean o
  WHERE o.order_status = 'delivered'
),
ventas_por_categoria AS (
  -- Ventas por categoría desde order_items (granularidad de item)
  SELECT
    COALESCE(p.product_category_clean, 'sin_categoria') AS categoria,
    oi.order_id,
    SUM(oi.total_item_value) AS venta_pedido  -- Suma de items por pedido
  FROM workspace.lumi_silver.order_items_clean oi
  INNER JOIN order_base o
    ON oi.order_id = o.order_id
  LEFT JOIN workspace.lumi_silver.products_clean p
    ON oi.product_id = p.product_id
  GROUP BY COALESCE(p.product_category_clean, 'sin_categoria'), oi.order_id
),
pagos_por_pedido AS (
  -- Pagos consolidados por pedido (payments_clean ya está consolidada)
  SELECT
    order_id,
    total_payment_value AS pago_total
  FROM workspace.lumi_silver.payments_clean
)
SELECT
  v.categoria,
  COUNT(DISTINCT v.order_id) AS pedidos,
  ROUND(SUM(v.venta_pedido), 2) AS venta_total,
  ROUND(AVG(v.venta_pedido), 2) AS ticket_promedio,
  ROUND(AVG(p.pago_total), 2) AS pago_promedio,
  ROUND(AVG(o.delay_days), 2) AS demora_promedio_dias,
  ROUND(100.0 * SUM(CASE WHEN o.is_late THEN 1 ELSE 0 END) / COUNT(DISTINCT v.order_id), 2) AS tasa_entrega_tarde_pct,
  ROUND(AVG(r.review_promedio_pedido), 2) AS review_promedio,
  SUM(CASE WHEN r.review_promedio_pedido < 3.0 THEN 1 ELSE 0 END) AS pedidos_review_bajo
FROM ventas_por_categoria v
INNER JOIN order_base o
  ON v.order_id = o.order_id
LEFT JOIN pagos_por_pedido p
  ON v.order_id = p.order_id
LEFT JOIN reviews_by_order r
  ON v.order_id = r.order_id
GROUP BY v.categoria
HAVING COUNT(DISTINCT v.order_id) >= 100
ORDER BY venta_total DESC
LIMIT 20;

### RETO CONSULTOR: 3 Insights Ejecutivos

---

**Insight 1: Demora en entregas deteriora satisfacción del cliente en 12.3%**

**Evidencia:**  
Pedidos entregados a tiempo: review promedio 4.23. Pedidos con +7 días tarde: 3.71 (-0.52 puntos, -12.3%).

**Impacto de negocio:**  
Reviews <4.0 aumentan churn y reducen conversión en marketplace. Categorías de alto valor pierden ventaja competitiva con entregas tardías.

**Recomendación:**  
1. Priorizar logística en health_beauty y watches_gifts (top ventas)
2. Alertas automáticas para pedidos en riesgo de +3 días
3. Compensación proactiva a pedidos con +5 días de retraso

**Consulta SQL:** Celda 29 (relación demora-review)

---

**Insight 2: SP concentra 40% de pedidos — riesgo de dependencia geográfica**

**Evidencia:**  
SP: 40,501 pedidos entregados (40% del total nacional). RJ + MG suman otros 23,704 pedidos (24%). Los 3 estados concentran 64% de pedidos. Estados del Norte y Centro-Oeste <2% cada uno.

**Impacto de negocio:**  
Dependencia excesiva de una región. Crisis económica o logística en SP impacta 40% de ingresos. Oportunidad desaprovechada en regiones con baja penetración.

**Recomendación:**  
1. Lanzar campaña de expansión en estados con <1,000 pedidos/año
2. Asociarse con húbs logísticos en Norte y Centro-Oeste
3. Incentivos de envío gratuito para primeras compras en regiones objetivo
4. Meta: reducir dependencia de SP a <35% en 12 meses

**Consulta SQL:** Celda 53 (top estados por pedidos entregados)

---

**Insight 3: Lluvia reduce producción de bagazo 23-28% — necesidad de buffer estratégico**

**Evidencia:**  
Días sin lluvia alta: 455-565 ton bagazo promedio. Días con lluvia alta: 327-451 ton (-23% a -28%). Correlación lluvia-bagazo: -0.25 a -0.33.

**Impacto de negocio:**  
Pérdida de 100-150 ton diarias de bagazo para energía. Compra urgente de combustible alternativo (30-40% más caro). Incumplimiento potencial de contratos de suministro energético.

**Recomendación:**  
1. Buffer estratégico de 1,500 ton bagazo (equivalente a 3-4 días lluviosos)
2. Contratos flexibles con proveedores de combustible alternativo
3. Integrar pronósticos meteorológicos SENAMHI en planificación semanal
4. Excluir días de lluvia alta de modelos ML de forecasting

**Consulta SQL:** Celdas 44, 46 (días secos vs lluviosos, correlaciones)